# Dev notebook — phase2_aqi.py

Chạy từng bước Pha 2 để xem/hiểu Nowcast + `iaqi_hour()` hoạt động thế nào trên output thật của Pha 1. Notebook này **import thẳng từ `phase2_aqi.py`** — không định nghĩa lại logic.

Cần chạy `phase1_dev.ipynb` (hoặc `python jobs/phase1_clean.py ...`) trước để có input ở `/tmp/clean`.

Sửa `phase2_aqi.py` xong, chạy lại cell là thấy ngay nhờ `%autoreload 2`.

In [5]:
%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

from phase2_aqi import add_nowcast, compute_iaqi_hour, print_owm_comparison

spark = (
    SparkSession.builder.appName("phase2_dev").master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")  # bắt buộc, xem ghi chú trong phase1_clean.py
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

CLEAN_INPUT = "/tmp/clean"  # output của phase1_clean.py — chạy phase1 trước nếu chưa có

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Bước 0 — đọc output Pha 1

In [6]:
df = spark.read.parquet(CLEAN_INPUT)
print("so dong:", df.count())
df.select("station_id", "ts_utc", "pm2_5", "pm10", "o3", "no2", "so2", "co", "owm_aqi").show(5)

so dong: 10800
+----------+-------------------+-----+-----+-----+----+----+------+-------+
|station_id|             ts_utc|pm2_5| pm10|   o3| no2| so2|    co|owm_aqi|
+----------+-------------------+-----+-----+-----+----+----+------+-------+
| VN_BDU_01|2026-07-11 00:00:00| 5.82| 8.47| 18.5|3.36|0.43|317.78|      1|
| VN_BDU_01|2026-07-11 01:00:00| 4.18| 5.21|32.27|3.88| 1.6|  80.0|      1|
| VN_BDU_01|2026-07-11 02:00:00| 4.74| 6.64|31.36|6.74| 0.3|141.37|      1|
| VN_BDU_01|2026-07-11 03:00:00| 4.95|  7.3| 45.0| 1.2| 0.8|615.83|      1|
| VN_BDU_01|2026-07-11 04:00:00|  2.3|4.945|55.07|2.26|0.62|617.33|      1|
+----------+-------------------+-----+-----+-----+----+----+------+-------+
only showing top 5 rows



## Bước 1 — `add_nowcast()`: xem Nowcast tính ra sao trên 1 trạm cụ thể

In [7]:
with_nc = add_nowcast(df, "pm2_5")
with_nc = add_nowcast(with_nc, "pm10")

one_station = with_nc.filter(F.col("station_id") == "VN_HCM_01").orderBy("ts_epoch")
one_station.select("ts_epoch", "pm2_5", "pm2_5_nowcast").show(15)

+----------+-----+------------------+
|  ts_epoch|pm2_5|     pm2_5_nowcast|
+----------+-----+------------------+
|1781481600| 6.76|              NULL|
|1781485200| 5.56| 6.101558441558441|
|1781488800| 7.66| 6.772865300638227|
|1781492400| 6.55| 6.688290369895664|
|1781496000| 6.91| 6.764408624624466|
|1781499600| 7.49|6.9974046638443665|
|1781503200| 5.27|6.4340421591537345|
|1781506800| 4.42| 5.501878984443797|
|1781510400| 4.17| 4.848370364978515|
|1781514000| 5.17| 4.995244946152579|
|1781517600| 5.42| 5.189010248316823|
|1781521200| 7.96| 6.507823638286572|
|1781524800| 9.19| 7.882344322344323|
|1781528400| 9.22| 8.551619047619049|
|1781532000| 9.51| 9.031035409035411|
+----------+-----+------------------+
only showing top 15 rows



So sánh cột `pm2_5` (đo thô 1h) với `pm2_5_nowcast` (trung bình trọng số 12h) — Nowcast sẽ "mượt" hơn, đi theo xu hướng chứ không nhảy giật theo từng giờ.

## Bước 2 — `compute_iaqi_hour()`: Nowcast + Reducer AQI giờ

In [8]:
result = compute_iaqi_hour(df)
print("so dong co aqi:", result.filter(F.col("aqi").isNotNull()).count(), "/", result.count())

result.filter(F.col("station_id") == "VN_HCM_01").orderBy("ts_epoch").select(
    "ts_utc", "pm2_5_nowcast", "pm10_nowcast", "aqi", "aqi_level", "aqi_label", "dominant_pollutant",
).show(15)

so dong co aqi: 10145 / 10800
+-------------------+------------------+------------------+----+---------+---------+------------------+
|             ts_utc|     pm2_5_nowcast|      pm10_nowcast| aqi|aqi_level|aqi_label|dominant_pollutant|
+-------------------+------------------+------------------+----+---------+---------+------------------+
|2026-06-15 00:00:00|              NULL|              NULL| 7.0|        1|      Tốt|                o3|
|2026-06-15 01:00:00| 6.101558441558441| 8.135842763549732|12.0|        1|      Tốt|             pm2_5|
|2026-06-15 02:00:00| 6.772865300638227| 9.348035313365513|14.0|        1|      Tốt|             pm2_5|
|2026-06-15 03:00:00| 6.688290369895664| 9.481611166461144|13.0|        1|      Tốt|             pm2_5|
|2026-06-15 04:00:00| 6.764408624624466| 9.285783473273348|14.0|        1|      Tốt|             pm2_5|
|2026-06-15 05:00:00|6.9974046638443665|10.095658896843727|18.0|        1|      Tốt|                o3|
|2026-06-15 06:00:00|6.43404215915

## Bước 3 — vì sao vài giờ bị `qc=missing` (Pha 1) vẫn có `aqi` (Pha 2)?

Nowcast chỉ cần >=2/3 giá trị trong 3 giờ gần nhất — nên GIỜ ĐẦU TIÊN của 1 khối gap vẫn
có thể ước lượng được từ vài giờ liền trước, dù bản thân giờ đó bị đánh dấu thiếu hẳn.

In [9]:
edge_case = result.filter((F.col("pm2_5_qc") == "missing") & F.col("aqi").isNotNull())
print("so gio 'missing' nhung van co aqi (Nowcast tu du lieu con lai):", edge_case.count())
edge_case.select("station_id", "ts_utc", "pm2_5_qc", "pm2_5_nowcast", "aqi").show(5)

so gio 'missing' nhung van co aqi (Nowcast tu du lieu con lai): 17
+----------+-------------------+--------+------------------+-----+
|station_id|             ts_utc|pm2_5_qc|     pm2_5_nowcast|  aqi|
+----------+-------------------+--------+------------------+-----+
| IN_DEL_01|2026-07-11 01:00:00| missing| 44.99576288719322| 90.0|
| IN_DEL_01|2026-07-23 01:00:00| missing|188.51820713238888|239.0|
| IN_DEL_01|2026-07-31 01:00:00| missing| 49.88239863214459|100.0|
| IN_DEL_01|2026-08-10 01:00:00| missing| 55.14302882266731|109.0|
| JP_TYO_01|2026-07-07 01:00:00| missing|3.5100488519785054|  7.0|
+----------+-------------------+--------+------------------+-----+
only showing top 5 rows



## Bước 4 — bảng đối chiếu owm_aqi

In [10]:
_ = print_owm_comparison(result)


=== Đối chiếu AQI tự tính vs owm_aqi (OpenWeather, thang 1-5) ===
Số bản ghi có cả 2 giá trị: 10128


+-------+----+------------+------------+------------+
|owm_aqi|   n|avg_aqi_tinh|min_aqi_tinh|max_aqi_tinh|
+-------+----+------------+------------+------------+
|      1|4630|        13.2|         3.0|        30.0|
|      2|2854|        31.5|         8.0|        83.0|
|      3|1515|        70.0|         8.0|       138.0|
|      4| 525|       116.5|        14.0|       170.0|
|      5| 604|       174.8|         9.0|       359.0|
+-------+----+------------+------------+------------+




## Ghi chú

- Notebook này chỉ để đọc/hiểu/debug — sửa logic thì sửa `phase2_aqi.py`.
- Muốn ghi thử output thật ra parquet: gọi lại đúng đoạn `main()` trong `phase2_aqi.py`, hoặc chạy CLI `python jobs/phase2_aqi.py --input /tmp/clean --output /tmp/aqi`.